# MLFlow Project Tracking

In [ ]:
import os
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
client = mlflow.MlflowClient()

PROJECT_DIR = "iris_project"
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project directory:", os.path.abspath(PROJECT_DIR))

## Data generation and storage

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

data_path = os.path.join(PROJECT_DIR, 'data')
os.makedirs(data_path, exist_ok=True)
X_train.to_csv(os.path.join(data_path, "X_train.csv"), index=False)
X_test.to_csv(os.path.join(data_path, "X_test.csv"), index=False)
y_train.to_csv(os.path.join(data_path, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(data_path, "y_test.csv"), index=False)

## Train script

In [ ]:
%%writefile iris_project/train.py
import os
import argparse
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--n_estimators", type=int, default=3)
    parser.add_argument("--max_depth", type=int, default=3)
    parser.add_argument("--data_path", type=str, default="data")
    args = parser.parse_args()

    mlflow.set_tracking_uri("http://localhost:5000")

    X_train = pd.read_csv(os.path.join(args.data_path, "X_train.csv"))
    X_test = pd.read_csv(os.path.join(args.data_path, "X_test.csv"))
    y_train = pd.read_csv(os.path.join(args.data_path, "y_train.csv")).values.ravel()
    y_test = pd.read_csv(os.path.join(args.data_path, "y_test.csv")).values.ravel()

    with mlflow.start_run(run_name=f"project-run-n{args.n_estimators}-d{args.max_depth}"):
        mlflow.log_param("n_estimators", args.n_estimators)
        mlflow.log_param("max_depth", args.max_depth)

        model = RandomForestClassifier(
            n_estimators=args.n_estimators, max_depth=args.max_depth, random_state=42
        )
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds, average="macro")
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_macro", f1)
        mlflow.sklearn.log_model(model, name="model")

        print(f"accuracy={acc:.4f}  f1_macro={f1:.4f}  run_id={mlflow.active_run().info.run_id}")


if __name__ == "__main__":
    main()

## YAML Files

In [ ]:
%%writefile iris_project/MLproject
name: iris-classifier

python_env: python_env.yaml

entry_points:
  main:
    parameters:
      n_estimators: {type: int, default: 3}
      max_depth: {type: int, default: 3}
      data_path: {type: str, default: "data"}
    command: "python train.py --n_estimators {n_estimators} --max_depth {max_depth} --data_path {data_path}"

In [ ]:
%%writefile iris_project/python_env.yaml
python: "3.12"
build_dependencies:
  - pip
  - setuptools
dependencies:
  - scikit-learn
  - pandas
  - mlflow

## Main run

In [ ]:
!mlflow run iris_project -P n_estimators=3 -P max_depth=3 \
    --experiment-name iris-classifier --env-manager=local

## Register the model and transition to Staging

In [ ]:
import mlflow
runs_df = mlflow.search_runs(
    experiment_names=["iris-classifier"],
    order_by=["start_time DESC"],
)
run_id = runs_df['run_id'][0]
print(run_id)

In [ ]:
result = mlflow.register_model(model_uri=f"runs:/{run_id}/model", name="iris-rf")
client.transition_model_version_stage(name=result.name, version=result.version, stage="Staging")